# Amazon ML Challenge 2026 — Production Inference & Submission

Production test-inference notebook after blocking.

Locked baseline:
- V2 normalization
- token-frequency threshold = `1000`
- top-K rare tokens = `2`
- LightGBM matcher with the current 13 features
- current experimental decision threshold = `0.998`

`0.998` is configurable and should be replaced if the corrected validation experiment finds a better threshold.

This notebook is chunked and restartable for a 16 GB RAM machine.


In [7]:
from pathlib import Path

import re
import gc
import json
import math
import pickle
import time
import subprocess
import sys

from collections import Counter, defaultdict

import numpy as np
import pandas as pd

from rapidfuzz.fuzz import ratio, WRatio


# ============================================================
# PROJECT ROOT
# Notebook location:
# D:\amzn_ml_challenge\notebooks
#
# Therefore project root is:
# D:\amzn_ml_challenge
# ============================================================

ROOT = Path.cwd().parent


# ============================================================
# DATA DIRECTORIES
# ============================================================

DATA_DIR = ROOT / "data"

RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"


# ============================================================
# MODEL / OUTPUT DIRECTORIES
# ============================================================

MODEL_DIR = ROOT / "models"

OUTPUT_DIR = ROOT / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHUNK_DIR = OUTPUT_DIR / "inference_chunks"
CHUNK_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# LOCKED BLOCKING CONFIGURATION
# ============================================================

TOKEN_FREQ_THRESHOLD = 1000
TOP_K = 2


# ============================================================
# MATCHER CONFIGURATION
# ============================================================

MATCH_THRESHOLD = 0.998


# ============================================================
# INFERENCE CONFIGURATION
# ============================================================

S1_CHUNK_SIZE = 10_000


# ============================================================
# MODEL PATHS
# ============================================================

MODEL_PATH = MODEL_DIR / "entity_matcher_lightgbm.pkl"
MODEL_CONFIG_PATH = MODEL_DIR / "matcher_config.pkl"


# ============================================================
# OUTPUT PATHS
# ============================================================

RESULT_PATH = OUTPUT_DIR / "matching_results.tsv"


# ============================================================
# VERIFY PROJECT ROOT
# ============================================================

print("=" * 70)
print("AMAZON ML CHALLENGE 2026 — INFERENCE CONFIGURATION")
print("=" * 70)

print(f"\nROOT:            {ROOT}")

print("\nDATA:")
print(f"RAW:             {RAW_DIR}")
print(f"INTERIM:         {INTERIM_DIR}")
print(f"PROCESSED:       {PROCESSED_DIR}")

print("\nMODEL:")
print(f"MODEL:           {MODEL_PATH}")
print(f"CONFIG:          {MODEL_CONFIG_PATH}")

print("\nOUTPUT:")
print(f"OUTPUT:          {OUTPUT_DIR}")
print(f"CHUNKS:          {CHUNK_DIR}")
print(f"RESULT:          {RESULT_PATH}")

print("\nBLOCKING:")
print(f"TOKEN_THRESHOLD: {TOKEN_FREQ_THRESHOLD}")
print(f"TOP_K:           {TOP_K}")

print("\nMATCHER:")
print(f"THRESHOLD:       {MATCH_THRESHOLD}")

print("\nINFERENCE:")
print(f"S1_CHUNK_SIZE:   {S1_CHUNK_SIZE}")

print("=" * 70)

AMAZON ML CHALLENGE 2026 — INFERENCE CONFIGURATION

ROOT:            d:\amzn_ml_challenge

DATA:
RAW:             d:\amzn_ml_challenge\data\raw
INTERIM:         d:\amzn_ml_challenge\data\interim
PROCESSED:       d:\amzn_ml_challenge\data\processed

MODEL:
MODEL:           d:\amzn_ml_challenge\models\entity_matcher_lightgbm.pkl
CONFIG:          d:\amzn_ml_challenge\models\matcher_config.pkl

OUTPUT:
OUTPUT:          d:\amzn_ml_challenge\output
CHUNKS:          d:\amzn_ml_challenge\output\inference_chunks
RESULT:          d:\amzn_ml_challenge\output\matching_results.tsv

BLOCKING:
TOKEN_THRESHOLD: 1000
TOP_K:           2

MATCHER:
THRESHOLD:       0.998

INFERENCE:
S1_CHUNK_SIZE:   10000


In [9]:
print("ROOT EXISTS:", ROOT.exists())
print("RAW EXISTS:", RAW_DIR.exists())
print("MODEL EXISTS:", MODEL_DIR.exists())

print("\nRaw files:")
for f in sorted(RAW_DIR.glob("*.tsv")):
    print(" ", f.name)

print("\nModel files:")
for f in sorted(MODEL_DIR.glob("*")):
    print(" ", f.name)

ROOT EXISTS: True
RAW EXISTS: True
MODEL EXISTS: True

Raw files:
  test_source1.tsv
  test_source2.tsv
  test_source3.tsv
  train_ground_truth.tsv
  train_source1.tsv
  train_source2.tsv
  train_source3.tsv

Model files:
  entity_matcher_lightgbm.pkl
  matcher_config.pkl


## 1. Load matcher model

In [10]:
if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Matcher model not found: {MODEL_PATH}. "
        "Run the matcher-training notebook first."
    )

# Load LightGBM matcher
with open(MODEL_PATH, "rb") as f:
    model = pickle.load(f)

print("Loaded model:", type(model).__name__)


# Load matcher configuration
if not MODEL_CONFIG_PATH.exists():
    raise FileNotFoundError(
        f"Matcher config not found: {MODEL_CONFIG_PATH}"
    )

with open(MODEL_CONFIG_PATH, "rb") as f:
    matcher_config = pickle.load(f)

print("Loaded config:", type(matcher_config).__name__)

print("\nMatcher configuration:")
print(matcher_config)

Loaded model: LGBMClassifier
Loaded config: dict

Matcher configuration:
{'random_state': 42, 'blocking': {'normalization': 'V2', 'token_frequency_threshold': 1000, 'top_k': 2}, 'matcher_features': ['country_same', 'name_exact', 'address_exact', 'name_length_diff', 'address_length_diff', 'name_token_jaccard', 'name_token_containment', 'address_token_jaccard', 'address_token_containment', 'name_char_ratio', 'address_char_ratio', 'name_wratio', 'address_wratio'], 'recommended_threshold_from_current_10k_experiment': 0.998}


## 2. Load test data

In [11]:

test_s1 = pd.read_csv(RAW_DIR / "test_source1.tsv", sep="\t", dtype=str)
test_s2 = pd.read_csv(RAW_DIR / "test_source2.tsv", sep="\t", dtype=str)
test_s3 = pd.read_csv(RAW_DIR / "test_source3.tsv", sep="\t", dtype=str)

required = {"entity_id", "business_name", "business_address", "country"}

for name, df in [("S1", test_s1), ("S2", test_s2), ("S3", test_s3)]:
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{name} missing: {missing}")
    if df["entity_id"].duplicated().any():
        raise ValueError(f"{name} has duplicate entity_id values")

print("S1:", test_s1.shape)
print("S2:", test_s2.shape)
print("S3:", test_s3.shape)


S1: (1732544, 4)
S2: (4887273, 4)
S3: (5082316, 4)


## 3. Locked V2 normalization

In [12]:

import unicodedata

def normalize_name_v2(value):
    if pd.isna(value):
        return ""
    text = str(value)
    text = unicodedata.normalize("NFKC", text)
    text = unicodedata.normalize("NFKD", text)
    text = "".join(c for c in text if not unicodedata.combining(c))
    text = text.casefold()
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def normalize_address_v2(value):
    if pd.isna(value):
        return ""
    text = str(value)
    text = unicodedata.normalize("NFKC", text)
    text = unicodedata.normalize("NFKD", text)
    text = "".join(c for c in text if not unicodedata.combining(c))
    text = text.casefold()
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)

    replacements = {
        r"\bave\b":"avenue", r"\bav\b":"avenue",
        r"\bst\b":"street", r"\brd\b":"road", r"\bdr\b":"drive",
        r"\bln\b":"lane", r"\btrl\b":"trail", r"\bhwy\b":"highway",
        r"\bblvd\b":"boulevard", r"\bpkwy\b":"parkway",
        r"\bct\b":"court", r"\bpl\b":"place", r"\bcir\b":"circle",
        r"\bter\b":"terrace", r"\bste\b":"suite", r"\bapt\b":"apartment",
    }
    for pattern, replacement in replacements.items():
        text = re.sub(pattern, replacement, text)
    return re.sub(r"\s+", " ", text).strip()

for df in (test_s1, test_s2, test_s3):
    df["normalized_name_v2"] = df["business_name"].map(normalize_name_v2)
    df["normalized_address_v2"] = df["business_address"].map(normalize_address_v2)

print("Normalization complete.")


Normalization complete.


## 4. Build locked V2 indexes

Rarity follows the locked blocker: frequencies are computed from Source 2 and the resulting rare tokens are used against both S2 and S3 indexes.


In [13]:

def build_exact_index(df, column):
    idx = defaultdict(list)
    for value, entity_id in zip(df[column].to_numpy(), df["entity_id"].to_numpy()):
        if value:
            idx[value].append(entity_id)
    return idx

def build_token_index(df, column):
    idx = defaultdict(list)
    freq = Counter()
    values = df[column].to_numpy()
    ids = df["entity_id"].to_numpy()

    for value in values:
        if value:
            freq.update(set(value.split()))

    for value, entity_id in zip(values, ids):
        if value:
            for token in set(value.split()):
                idx[token].append(entity_id)
    return idx, freq

s2_name_exact = build_exact_index(test_s2, "normalized_name_v2")
s2_address_exact = build_exact_index(test_s2, "normalized_address_v2")
s3_name_exact = build_exact_index(test_s3, "normalized_name_v2")
s3_address_exact = build_exact_index(test_s3, "normalized_address_v2")

s2_name_index, s2_name_freq = build_token_index(test_s2, "normalized_name_v2")
s2_address_index, s2_address_freq = build_token_index(test_s2, "normalized_address_v2")
s3_name_index, _ = build_token_index(test_s3, "normalized_name_v2")
s3_address_index, _ = build_token_index(test_s3, "normalized_address_v2")

print("Indexes built.")
print("S2 name tokens:", len(s2_name_index))
print("S2 address tokens:", len(s2_address_index))
print("S3 name tokens:", len(s3_name_index))
print("S3 address tokens:", len(s3_address_index))
gc.collect()


Indexes built.
S2 name tokens: 669307
S2 address tokens: 604075
S3 name tokens: 712682
S3 address tokens: 575653


0

## 5. Locked blocker

In [14]:

def generate_v2_block_candidates_1000(s1_row, top_k=TOP_K):
    name = s1_row["normalized_name_v2"]
    address = s1_row["normalized_address_v2"]

    s2_candidates = set()
    s3_candidates = set()

    if name:
        s2_candidates.update(s2_name_exact.get(name, []))
        s3_candidates.update(s3_name_exact.get(name, []))

    if address:
        s2_candidates.update(s2_address_exact.get(address, []))
        s3_candidates.update(s3_address_exact.get(address, []))

    if name:
        eligible = [
            t for t in set(name.split())
            if s2_name_freq.get(t, 10**9) <= TOKEN_FREQ_THRESHOLD
        ]
        for token in sorted(
            eligible, key=lambda t: s2_name_freq.get(t, 10**9)
        )[:top_k]:
            s2_candidates.update(s2_name_index.get(token, []))
            s3_candidates.update(s3_name_index.get(token, []))

    if address:
        eligible = [
            t for t in set(address.split())
            if s2_address_freq.get(t, 10**9) <= TOKEN_FREQ_THRESHOLD
        ]
        for token in sorted(
            eligible, key=lambda t: s2_address_freq.get(t, 10**9)
        )[:top_k]:
            s2_candidates.update(s2_address_index.get(token, []))
            s3_candidates.update(s3_address_index.get(token, []))

    return list(s2_candidates) + list(s3_candidates)

sample_candidates = generate_v2_block_candidates_1000(test_s1.iloc[0])
print("Smoke candidate count:", len(sample_candidates))


Smoke candidate count: 610


## 6. Target lookup frames

In [16]:
def target_frame(df):

    return df[
        [
            "entity_id",
            "business_name",
            "business_address",
            "country",
            "normalized_name_v2",
            "normalized_address_v2",
        ]
    ].rename(columns={
        "entity_id": "target_entity_id",
        "business_name": "target_business_name",
        "business_address": "target_business_address",
        "country": "target_country",
        "normalized_name_v2": "target_name_v2",
        "normalized_address_v2": "target_address_v2",
    })


s2_target = target_frame(test_s2)
s3_target = target_frame(test_s3)

print("S2 target rows:", len(s2_target))
print("S3 target rows:", len(s3_target))

S2 target rows: 4887273
S3 target rows: 5082316


## 7. Matcher features

In [17]:

def token_jaccard(a, b):
    if not a or not b:
        return 0.0
    sa, sb = set(a.split()), set(b.split())
    union = sa | sb
    return len(sa & sb) / len(union) if union else 0.0

def token_containment(a, b):
    if not a or not b:
        return 0.0
    sa, sb = set(a.split()), set(b.split())
    m = min(len(sa), len(sb))
    return len(sa & sb) / m if m else 0.0

FEATURE_COLS = [
    "country_same","name_exact","address_exact",
    "name_length_diff","address_length_diff",
    "name_token_jaccard","name_token_containment",
    "address_token_jaccard","address_token_containment",
    "name_char_ratio","address_char_ratio",
    "name_wratio","address_wratio",
]

def build_feature_frame(df):
    sn = df["s1_name_v2"].fillna("")
    sa = df["s1_address_v2"].fillna("")
    tn = df["target_name_v2"].fillna("")
    ta = df["target_address_v2"].fillna("")

    f = pd.DataFrame(index=df.index)
    f["country_same"] = (
        df["s1_country"].fillna("") == df["target_country"].fillna("")
    ).astype("int8")
    f["name_exact"] = (sn == tn).astype("int8")
    f["address_exact"] = (sa == ta).astype("int8")
    f["name_length_diff"] = (sn.str.len() - tn.str.len()).abs().astype("float32")
    f["address_length_diff"] = (sa.str.len() - ta.str.len()).abs().astype("float32")

    f["name_token_jaccard"] = [token_jaccard(a,b) for a,b in zip(sn,tn)]
    f["name_token_containment"] = [token_containment(a,b) for a,b in zip(sn,tn)]
    f["address_token_jaccard"] = [token_jaccard(a,b) for a,b in zip(sa,ta)]
    f["address_token_containment"] = [token_containment(a,b) for a,b in zip(sa,ta)]

    f["name_char_ratio"] = [ratio(a,b)/100 for a,b in zip(sn,tn)]
    f["address_char_ratio"] = [ratio(a,b)/100 for a,b in zip(sa,ta)]
    f["name_wratio"] = [WRatio(a,b)/100 for a,b in zip(sn,tn)]
    f["address_wratio"] = [WRatio(a,b)/100 for a,b in zip(sa,ta)]
    return f


## 8. Chunk inference

Only accepted pairs are returned from each chunk. Candidate matrices are discarded before moving to the next chunk.


In [21]:
def infer_chunk(s1_chunk):
    """
    Memory-safe inference.

    Processes the supplied S1 chunk in small batches so that
    candidate pairs are never materialized for the entire chunk.
    """

    outputs = []

    n_batches = math.ceil(len(s1_chunk) / INFERENCE_BATCH_SIZE)

    for batch_idx in range(n_batches):

        batch_start = batch_idx * INFERENCE_BATCH_SIZE
        batch_end = min(
            batch_start + INFERENCE_BATCH_SIZE,
            len(s1_chunk)
        )

        batch = s1_chunk.iloc[batch_start:batch_end].copy()

        # ----------------------------------------------------
        # Generate candidates ONLY for this small batch
        # ----------------------------------------------------

        rows = []

        for row in batch.itertuples(index=False):

            candidates = generate_v2_block_candidates_1000(
                row,
                top_k=TOP_K
            )

            if candidates:
                rows.extend(
                    (row.entity_id, target_id)
                    for target_id in candidates
                )

        # ----------------------------------------------------
        # No candidates
        # ----------------------------------------------------

        if not rows:
            del batch, rows
            gc.collect()
            continue

        candidate_pairs = pd.DataFrame(
            rows,
            columns=[
                "source1_entity_id",
                "target_entity_id"
            ]
        )

        del rows

        # ----------------------------------------------------
        # Attach S1 fields
        # ----------------------------------------------------

        candidate_features = candidate_pairs.merge(
            batch[
                [
                    "entity_id",
                    "business_name",
                    "business_address",
                    "country",
                    "normalized_name_v2",
                    "normalized_address_v2",
                ]
            ].rename(columns={
                "entity_id": "source1_entity_id",
                "business_name": "source1_business_name",
                "business_address": "source1_business_address",
                "country": "source1_country",
                "normalized_name_v2": "source1_name_v2",
                "normalized_address_v2": "source1_address_v2",
            }),
            on="source1_entity_id",
            how="left",
            sort=False,
        )

        del candidate_pairs

        # ----------------------------------------------------
        # Split S2 / S3 candidates
        # ----------------------------------------------------

        s2_mask = candidate_features["target_entity_id"].str.startswith(
            "S2-"
        )

        s3_mask = ~s2_mask

        s2_candidates = candidate_features.loc[s2_mask].copy()
        s3_candidates = candidate_features.loc[s3_mask].copy()

        del candidate_features

        # ----------------------------------------------------
        # Join target information
        # ----------------------------------------------------

        if not s2_candidates.empty:

            s2_candidates = s2_candidates.merge(
                s2_target,
                on="target_entity_id",
                how="left",
                sort=False,
            )

        if not s3_candidates.empty:

            s3_candidates = s3_candidates.merge(
                s3_target,
                on="target_entity_id",
                how="left",
                sort=False,
            )

        # ----------------------------------------------------
        # Combine again
        # ----------------------------------------------------

        feature_parts = []

        if not s2_candidates.empty:
            feature_parts.append(s2_candidates)

        if not s3_candidates.empty:
            feature_parts.append(s3_candidates)

        if not feature_parts:
            del batch
            gc.collect()
            continue

        candidate_features = pd.concat(
            feature_parts,
            ignore_index=True
        )

        del s2_candidates, s3_candidates, feature_parts

        # ----------------------------------------------------
        # Cheap features
        # ----------------------------------------------------

        candidate_features["country_same"] = (
            candidate_features["source1_country"]
            == candidate_features["target_country"]
        ).astype(np.int8)

        candidate_features["name_exact"] = (
            candidate_features["source1_name_v2"]
            == candidate_features["target_name_v2"]
        ).astype(np.int8)

        candidate_features["address_exact"] = (
            candidate_features["source1_address_v2"]
            == candidate_features["target_address_v2"]
        ).astype(np.int8)

        candidate_features["name_length_diff"] = (
            candidate_features["source1_name_v2"].str.len()
            - candidate_features["target_name_v2"].str.len()
        ).abs().astype(np.int16)

        candidate_features["address_length_diff"] = (
            candidate_features["source1_address_v2"].str.len()
            - candidate_features["target_address_v2"].str.len()
        ).abs().astype(np.int16)

        # ----------------------------------------------------
        # Token features
        # ----------------------------------------------------

        candidate_features["name_token_jaccard"] = [
            token_jaccard(a, b)
            for a, b in zip(
                candidate_features["source1_name_v2"],
                candidate_features["target_name_v2"],
            )
        ]

        candidate_features["name_token_containment"] = [
            token_containment(a, b)
            for a, b in zip(
                candidate_features["source1_name_v2"],
                candidate_features["target_name_v2"],
            )
        ]

        candidate_features["address_token_jaccard"] = [
            token_jaccard(a, b)
            for a, b in zip(
                candidate_features["source1_address_v2"],
                candidate_features["target_address_v2"],
            )
        ]

        candidate_features["address_token_containment"] = [
            token_containment(a, b)
            for a, b in zip(
                candidate_features["source1_address_v2"],
                candidate_features["target_address_v2"],
            )
        ]

        # ----------------------------------------------------
        # Character similarity
        # ----------------------------------------------------

        candidate_features["name_char_ratio"] = [
            ratio(a, b) / 100.0
            for a, b in zip(
                candidate_features["source1_name_v2"],
                candidate_features["target_name_v2"],
            )
        ]

        candidate_features["address_char_ratio"] = [
            ratio(a, b) / 100.0
            for a, b in zip(
                candidate_features["source1_address_v2"],
                candidate_features["target_address_v2"],
            )
        ]

        candidate_features["name_wratio"] = [
            WRatio(a, b) / 100.0
            for a, b in zip(
                candidate_features["source1_name_v2"],
                candidate_features["target_name_v2"],
            )
        ]

        candidate_features["address_wratio"] = [
            WRatio(a, b) / 100.0
            for a, b in zip(
                candidate_features["source1_address_v2"],
                candidate_features["target_address_v2"],
            )
        ]

        # ----------------------------------------------------
        # Model features
        # ----------------------------------------------------

        X = candidate_features[feature_cols_v2]

        probabilities = model.predict_proba(X)[:, 1]

        candidate_features["match_probability"] = probabilities

        # ----------------------------------------------------
        # Threshold
        # ----------------------------------------------------

        accepted = candidate_features.loc[
            candidate_features["match_probability"] >= MATCH_THRESHOLD,
            [
                "source1_entity_id",
                "target_entity_id",
                "match_probability",
            ]
        ].copy()

        if not accepted.empty:
            outputs.append(accepted)

        # ----------------------------------------------------
        # Aggressive cleanup
        # ----------------------------------------------------

        del X
        del candidate_features
        del batch

        gc.collect()

        print(
            f"    batch {batch_idx + 1}/{n_batches}: "
            f"candidates={len(probabilities):,} "
            f"accepted={len(accepted):,}"
        )

    # --------------------------------------------------------
    # Return accepted matches
    # --------------------------------------------------------

    if not outputs:
        return pd.DataFrame(
            columns=[
                "source1_entity_id",
                "target_entity_id",
                "match_probability",
            ]
        )

    return pd.concat(
        outputs,
        ignore_index=True
    )

## 9. Smoke test — run this before full inference

In [22]:
test_batch = test_s1.iloc[:250].copy()

t0 = time.time()

test_result = infer_chunk(test_batch)

print("\nTest complete")
print("Accepted matches:", len(test_result))
print("Time:", round(time.time() - t0, 2), "seconds")

display(test_result.head())

NameError: name 'INFERENCE_BATCH_SIZE' is not defined

In [19]:

smoke = infer_chunk(test_s1.iloc[:100].copy())
print("Smoke accepted pairs:", len(smoke))
print(smoke.head(20))


Smoke accepted pairs: 277
     source1_entity_id target_entity_id  match_probability
600       S1-156285671     S2-180394702           0.999153
603       S1-156285671     S2-971574924           0.999499
954       S1-689823050     S2-212601842           0.998883
970       S1-689823050     S2-829077756           0.999547
1008      S1-689823050     S2-319486263           0.998960
1353      S1-689823050      S2-65478038           0.999865
1544      S1-921369899     S2-808920974           0.999911
1545      S1-921369899     S2-995426671           0.999170
1548      S1-921369899     S2-107533354           0.999626
2026      S1-481669221     S2-284602321           0.999896
2349      S1-909865979     S2-528901108           0.999924
3211      S1-742053041     S2-446206310           0.998953
3220      S1-742053041      S2-73207920           0.999924
3223      S1-742053041     S2-315673501           0.998985
3228      S1-742053041     S2-317425677           0.999939
3230      S1-913506265     S2-

## 10. Full test inference

**Restartable:** existing completed chunk files are skipped.

If the IDE/kernel crashes:
1. restart kernel
2. rerun through Cell 9
3. leave `CLEAR_OLD_CHUNKS = False`
4. rerun this cell


In [20]:

CLEAR_OLD_CHUNKS = False

if CLEAR_OLD_CHUNKS:
    for p in CHUNK_DIR.glob("matches_*.tsv"):
        p.unlink()

n_chunks = math.ceil(len(test_s1) / S1_CHUNK_SIZE)

print("S1 rows:", len(test_s1))
print("Chunks:", n_chunks)

for chunk_idx in range(n_chunks):
    path = CHUNK_DIR / f"matches_{chunk_idx:05d}.tsv"

    if path.exists():
        print(f"[{chunk_idx+1}/{n_chunks}] SKIP {path.name}")
        continue

    start = chunk_idx * S1_CHUNK_SIZE
    end = min(start + S1_CHUNK_SIZE, len(test_s1))
    chunk = test_s1.iloc[start:end].copy()

    t0 = time.time()
    result = infer_chunk(chunk)
    result.to_csv(path, sep="\t", index=False)

    print(
        f"[{chunk_idx+1}/{n_chunks}] "
        f"S1={len(chunk):,} accepted={len(result):,} "
        f"time={time.time()-t0:.1f}s"
    )

    del chunk, result
    gc.collect()


S1 rows: 1732544
Chunks: 174


MemoryError: 

## 11. Combine accepted predictions

In [ ]:

chunk_files = sorted(CHUNK_DIR.glob("matches_*.tsv"))

if len(chunk_files) != n_chunks:
    missing = [
        i for i in range(n_chunks)
        if not (CHUNK_DIR / f"matches_{i:05d}.tsv").exists()
    ]
    raise RuntimeError(f"Missing chunks: {missing[:20]}")

parts = []
for path in chunk_files:
    df = pd.read_csv(path, sep="\t", dtype=str)
    if not df.empty:
        parts.append(df)

if parts:
    accepted_matches = pd.concat(parts, ignore_index=True)
else:
    accepted_matches = pd.DataFrame(
        columns=["source1_entity_id","target_entity_id","match_probability"]
    )

accepted_matches = accepted_matches.drop_duplicates(
    ["source1_entity_id","target_entity_id"]
)

print("Accepted pairs:", len(accepted_matches))
print("S1 entities with >=1 accepted pair:",
      accepted_matches["source1_entity_id"].nunique())


## 12. Prediction sanity checks

In [ ]:

if accepted_matches.duplicated(
    ["source1_entity_id","target_entity_id"]
).any():
    raise ValueError("Duplicate S1-target pairs found.")

if accepted_matches["target_entity_id"].str.startswith("S1-").any():
    raise ValueError("S1 ID appears as a target.")

if (~accepted_matches["target_entity_id"].isin(test_target_ids)).any():
    raise ValueError("Unknown target ID detected.")

print("Prediction checks passed.")


## 13. Write `matching_results.tsv`

In [ ]:

if accepted_matches.empty:
    grouped = pd.Series(dtype=str)
else:
    grouped = (
        accepted_matches
        .sort_values(["source1_entity_id","target_entity_id"])
        .groupby("source1_entity_id", sort=False)["target_entity_id"]
        .agg(",".join)
    )

matching_results = pd.DataFrame({
    "source1_entity_id": test_s1["entity_id"].astype(str)
})

matching_results["matched_entity_ids"] = (
    matching_results["source1_entity_id"]
    .map(grouped)
    .fillna("")
)

# Remove any accidental duplicate target IDs within a row.
matching_results["matched_entity_ids"] = matching_results[
    "matched_entity_ids"
].map(
    lambda x: ",".join(dict.fromkeys(x.split(","))) if x else ""
)

matching_results.to_csv(
    RESULT_PATH, sep="\t", index=False
)

print("Saved:", RESULT_PATH)
print("Rows:", len(matching_results))
print("Empty:", int((matching_results["matched_entity_ids"] == "").sum()))
print("Non-empty:", int((matching_results["matched_entity_ids"] != "").sum()))


## 14. Final structural validation

In [ ]:

if len(matching_results) != len(test_s1):
    raise ValueError("Wrong number of output rows.")

if set(matching_results["source1_entity_id"]) != set(test_s1["entity_id"]):
    raise ValueError("Output S1 IDs do not exactly match test S1 IDs.")

if matching_results["source1_entity_id"].duplicated().any():
    raise ValueError("Duplicate S1 rows.")

for value in matching_results.loc[
    matching_results["matched_entity_ids"] != "",
    "matched_entity_ids"
]:
    ids = value.split(",")
    if len(ids) != len(set(ids)):
        raise ValueError("Duplicate target inside prediction.")
    if any(x.startswith("S1-") for x in ids):
        raise ValueError("S1 target found.")

print("All structural checks passed.")


## 15. Official validator

The challenge documentation says to run `utils/validate_submission.py` before submission.

The exact CLI is repository-specific, so this cell first attempts the direct path form and prints the validator's output.


In [ ]:

validator = ROOT / "utils" / "validate_submission.py"

if not validator.exists():
    print("Validator not found:", validator)
else:
    proc = subprocess.run(
        [sys.executable, str(validator), str(RESULT_PATH)],
        cwd=ROOT,
        capture_output=True,
        text=True,
    )
    print("Return code:", proc.returncode)
    print(proc.stdout)
    print(proc.stderr)

    if proc.returncode != 0:
        print(
            "If the CLI expects different arguments, run:\n"
            f"  {sys.executable} {validator} --help"
        )


## 16. Final submission summary

In [ ]:

size_mb = RESULT_PATH.stat().st_size / (1024*1024)

print("=" * 60)
print("FINAL SUBMISSION")
print("=" * 60)
print("File:", RESULT_PATH)
print(f"Size: {size_mb:.2f} MB")
print("Rows:", len(matching_results))
print("Threshold:", MATCH_THRESHOLD)
print("Block threshold:", TOKEN_FREQ_THRESHOLD)
print("Block top-K:", TOP_K)
print("Accepted pairs:", len(accepted_matches))
print("Empty S1 predictions:",
      int((matching_results["matched_entity_ids"] == "").sum()))
print("\nFirst 10 rows:")
display(matching_results.head(10))


### Before leaderboard upload

The output file is:

```text
output/matching_results.tsv
```

Do not delete `output/inference_chunks/` until the result has passed validation.

If we later change the matcher threshold after proper validation, update `MATCH_THRESHOLD` and rerun inference. The chunk system will otherwise reuse old predictions, so delete the chunk files before rerunning with a different threshold.
